# Baseline: Markov Chain Music Generator

A 2nd-order Markov chain over pitches serves as one of the baselines against which the neural models are compared.

In [ ]:
import sys, os, random
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from collections import defaultdict, Counter
import pretty_midi

from src.config import cfg
from src.preprocessing import find_midi_files

## 1. Markov chain implementation

In [ ]:
class PitchMarkov:
    def __init__(self, order=2):
        self.order = order
        self.table = {}
    
    def fit(self, sequences):
        tmp = defaultdict(Counter)
        for seq in sequences:
            for i in range(len(seq) - self.order):
                tmp[tuple(seq[i:i+self.order])][seq[i+self.order]] += 1
        for k, c in tmp.items():
            total = sum(c.values())
            self.table[k] = (list(c.keys()), [v/total for v in c.values()])
    
    def generate(self, n=100, seed=None):
        if seed is None:
            seed = list(random.choice(list(self.table.keys())))
        out = list(seed)
        for _ in range(n):
            key = tuple(out[-self.order:])
            if key not in self.table:
                key = random.choice(list(self.table.keys()))
            toks, probs = self.table[key]
            out.append(np.random.choice(toks, p=probs))
        return out

## 2. Fit on training pitches

In [ ]:
files = find_midi_files(cfg.RAW_MIDI_DIR)[:50]
pitch_seqs = []
for f in files:
    try:
        pm = pretty_midi.PrettyMIDI(f)
        ps = [n.pitch for inst in pm.instruments
              for n in sorted(inst.notes, key=lambda x: x.start)]
        if len(ps) > 10:
            pitch_seqs.append(ps)
    except Exception:
        pass
print(f'Fitted on {len(pitch_seqs)} pitch sequences')

markov = PitchMarkov(order=2)
markov.fit(pitch_seqs)
print(f'Markov table size: {len(markov.table)} states')

## 3. Generate baseline MIDIs

In [ ]:
os.makedirs(cfg.MIDI_OUTPUT, exist_ok=True)

for i in range(5):
    pitches = markov.generate(100)
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)
    t = 0.0
    for p in pitches:
        dur = random.uniform(0.2, 0.6)
        inst.notes.append(pretty_midi.Note(
            velocity=80, pitch=int(p), start=t, end=t+dur))
        t += dur
    pm.instruments.append(inst)
    out_path = os.path.join(cfg.MIDI_OUTPUT, f'baseline_markov_{i+1}.mid')
    pm.write(out_path)
    print(f'Saved {out_path}')